In [1]:
import os
import functools
from glob import glob
from concurrent.futures import ThreadPoolExecutor

from dotenv import load_dotenv
from groq import Groq
import pymupdf  # pip install pymupdf

Load api key

In [2]:
load_dotenv()

client =Groq(api_key=os.getenv("GROQ_API_KEY"))

MODEL = "llama-3.3-70b-versatile"
EMBEDDING_MODEL = "text-embedding-3-small"
DB_NAME = "vector_db"

In [69]:
print(MODEL)

llama-3.3-70b-versatile


Đọc toàn bộ file pdf trong /data

In [83]:
from glob import glob
import os
import functools
import pymupdf

@functools.lru_cache(maxsize=None)
def load_context():
    context = {}
    pdf_files = [
        f for f in glob("data/*.pdf")
        if not os.path.basename(f).startswith("~$")
    ]

    for pdf_path in pdf_files:
        name = os.path.splitext(os.path.basename(pdf_path))[0]
        doc = pymupdf.open(pdf_path)
        text = ""
        for page in doc:
            text += page.get_text("text") + "\n"
        doc.close()
        context[name] = text

    return context

Test pdf

In [84]:
context = load_context()

print(context.keys())
for name, text in context.items():
    print("Tên tài liệu:", name)
    print("Độ dài text:", len(text))
    print(text[:1000])
    break

dict_keys(['Chí_Phèo'])
Tên tài liệu: Chí_Phèo
Độ dài text: 57779
1
Chí Phèo
Nam Cao
1941
Được lấy từ Wikisource vào ngày 26 tháng 3, 2026

2
ĐÔI LỨA XỨNG ĐÔI

Hắn vừa đi vừa chửi. Bao giờ cũng thế, cứ rượu xong là
hắn chửi. Bắt đầu hắn chửi giời. Có hề gì? giời có của riêng
nhà nào? Rồi hắn chửi đời. Thế cũng chẳng sao: đời là tất cả
nhưng chẳng là ai. Tức mình, hắn chửi ngay tất cả làng Vũ-
đại. Nhưng cả làng Vũ-đại, ai cũng tự nhủ: « Chắc nó trừ
mình ra! » Không ai lên tiếng cả. Tức thật! tức thật! ồ! thế
này thì tức thật! tức chết đi được mất! Đã thế, hắn phải chửi
cha đứa nào không chửi nhau với hắn. Nhưng cũng không ai
ra điều. Mẹ kiếp! thế thì có phí rượu không? Thế thì có khổ
hắn không? Không biết đứa chết mẹ nào lại đẻ ra thân hắn
cho hắn khổ đến nông-nỗi này? A hà! phải đấy, hắn cứ thế
mà chửi, hắn cứ chửi đứa chết mẹ nào đẻ ra thân hắn, đẻ ra
thằng Chí-Phèo! Hắn nghiến răng vào mà chửi cái đứa đã đẻ
ra Chí-Phèo. Nhưng mà biết đứa nào đã đẻ ra Chí-Phèo? Có
mà giời biết! Hắn k

context 

In [85]:
context = load_context()
print(f"Load context with {len(context)} documents: {list(context.keys())}")

Load context with 1 documents: ['Chí_Phèo']


In [86]:
context["Chí_Phèo"]

'1\nChí Phèo\nNam Cao\n1941\nĐược lấy từ Wikisource vào ngày 26 tháng 3, 2026\n\n2\nĐÔI LỨA XỨNG ĐÔI\n\nHắn vừa đi vừa chửi. Bao giờ cũng thế, cứ rượu xong là\nhắn chửi. Bắt đầu hắn chửi giời. Có hề gì? giời có của riêng\nnhà nào? Rồi hắn chửi đời. Thế cũng chẳng sao: đời là tất cả\nnhưng chẳng là ai. Tức mình, hắn chửi ngay tất cả làng Vũ-\nđại. Nhưng cả làng Vũ-đại, ai cũng tự nhủ: «\xa0Chắc nó trừ\nmình ra!\xa0» Không ai lên tiếng cả. Tức thật! tức thật! ồ! thế\nnày thì tức thật! tức chết đi được mất! Đã thế, hắn phải chửi\ncha đứa nào không chửi nhau với hắn. Nhưng cũng không ai\nra điều. Mẹ kiếp! thế thì có phí rượu không? Thế thì có khổ\nhắn không? Không biết đứa chết mẹ nào lại đẻ ra thân hắn\ncho hắn khổ đến nông-nỗi này? A hà! phải đấy, hắn cứ thế\nmà chửi, hắn cứ chửi đứa chết mẹ nào đẻ ra thân hắn, đẻ ra\nthằng Chí-Phèo! Hắn nghiến răng vào mà chửi cái đứa đã đẻ\nra Chí-Phèo. Nhưng mà biết đứa nào đã đẻ ra Chí-Phèo? Có\nmà giời biết! Hắn không biết, cả làng Vũ-đại cũng không

Định nghĩa vai trò

In [ ]:
system_message = {
    "role": "system",
    "content": """
Bạn là trợ lý hỏi đáp chỉ được phép trả lời dựa trên ngữ cảnh được cung cấp trong cuộc hội thoại.

Quy tắc bắt buộc:
- Chỉ dùng thông tin có trong phần "Ngữ cảnh".
- Tuyệt đối không dùng kiến thức bên ngoài.
- Nếu "Ngữ cảnh" không có thông tin để trả lời, phải trả lời đúng câu này:
  "Tôi không tìm thấy thông tin này trong tài liệu."
- Không suy đoán.
- Không tự bổ sung kiến thức phổ thông.
- Trả lời bằng tiếng Việt, ngắn gọn, trực tiếp.
- Với tác phẩm văn học, không dùng các từ như "sản xuất"; ưu tiên "viết", "ra đời", "xuất bản" nếu phù hợp với ngữ cảnh.
- Nếu câu hỏi hỏi về một tài liệu hoặc nhân vật không xuất hiện trong ngữ cảnh, vẫn phải trả lời:
"Tôi không tìm thấy thông tin này trong tài liệu."
"""
}

In [60]:
def get_relevant_context(message):
    relevant_context = []
    for context_title, context_details in context.items():
        if context_title.lower() in message.lower():
            relevant_context.append(context_details)
    return relevant_context 

nếu tên tài liệu xuất hiện trong câu hỏi → lấy context

In [61]:
get_relevant_context("Who is Quốc Chí")

[]

In [62]:
get_relevant_context("Ai là tác giả của Chí_Phèo")

['1\nChí Phèo\nNam Cao\n1941\nĐược lấy từ Wikisource vào ngày 26 tháng 3, 2026\n\n2\nĐÔI LỨA XỨNG ĐÔI\n\nHắn vừa đi vừa chửi. Bao giờ cũng thế, cứ rượu xong là\nhắn chửi. Bắt đầu hắn chửi giời. Có hề gì? giời có của riêng\nnhà nào? Rồi hắn chửi đời. Thế cũng chẳng sao: đời là tất cả\nnhưng chẳng là ai. Tức mình, hắn chửi ngay tất cả làng Vũ-\nđại. Nhưng cả làng Vũ-đại, ai cũng tự nhủ: «\xa0Chắc nó trừ\nmình ra!\xa0» Không ai lên tiếng cả. Tức thật! tức thật! ồ! thế\nnày thì tức thật! tức chết đi được mất! Đã thế, hắn phải chửi\ncha đứa nào không chửi nhau với hắn. Nhưng cũng không ai\nra điều. Mẹ kiếp! thế thì có phí rượu không? Thế thì có khổ\nhắn không? Không biết đứa chết mẹ nào lại đẻ ra thân hắn\ncho hắn khổ đến nông-nỗi này? A hà! phải đấy, hắn cứ thế\nmà chửi, hắn cứ chửi đứa chết mẹ nào đẻ ra thân hắn, đẻ ra\nthằng Chí-Phèo! Hắn nghiến răng vào mà chửi cái đứa đã đẻ\nra Chí-Phèo. Nhưng mà biết đứa nào đã đẻ ra Chí-Phèo? Có\nmà giời biết! Hắn không biết, cả làng Vũ-đại cũng khôn

In [64]:
get_relevant_context("ĐÔI LỨA XỨNG ĐÔI")

[]

Tạo giao diện chatbot đơn giản

In [94]:
def add_context(message):
    relevant_context = get_relevant_context(message)

    if not relevant_context:
        return f"""Câu hỏi: {message}

Ngữ cảnh: KHÔNG CÓ DỮ LIỆU"""
    
    context_text = "\n\n".join(relevant_context[:2])[:2500]

    return f"""Câu hỏi: {message}

Ngữ cảnh:
{context_text}"""

In [95]:
def chat(message, history):
    try:
        relevant_context = get_relevant_context(message)

        if not relevant_context:
            yield "Tôi không tìm thấy thông tin này trong tài liệu."
            return

        messages = [system_message]

        if history:
            for item in history[-6:]:
                if isinstance(item, dict):
                    role = item.get("role")
                    content = item.get("content")
                    if role and content:
                        messages.append({"role": role, "content": content})

        enriched_message = add_context(message)
        messages.append({"role": "user", "content": enriched_message})

        stream = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            stream=True,
            temperature=0,
            max_completion_tokens=300
        )

        response = ""
        for chunk in stream:
            delta = chunk.choices[0].delta.content
            if delta:
                response += delta
                yield response

    except Exception as e:
        yield f"Xin lỗi, đã có lỗi xảy ra: {e}"

In [91]:
ctx = add_context("ai là tác giả của tác phẩm Chí_Phèo")
print(ctx[:1000])

ai là tác giả của tác phẩm Chí_Phèo

Context:
1
Chí Phèo
Nam Cao
1941
Được lấy từ Wikisource vào ngày 26 tháng 3, 2026

2
ĐÔI LỨA XỨNG ĐÔI

Hắn vừa đi vừa chửi. Bao giờ cũng thế, cứ rượu xong là
hắn chửi. Bắt đầu hắn chửi giời. Có hề gì? giời có của riêng
nhà nào? Rồi hắn chửi đời. Thế cũng chẳng sao: đời là tất cả
nhưng chẳng là ai. Tức mình, hắn chửi ngay tất cả làng Vũ-
đại. Nhưng cả làng Vũ-đại, ai cũng tự nhủ: « Chắc nó trừ
mình ra! » Không ai lên tiếng cả. Tức thật! tức thật! ồ! thế
này thì tức thật! tức chết đi được mất! Đã thế, hắn phải chửi
cha đứa nào không chửi nhau với hắn. Nhưng cũng không ai
ra điều. Mẹ kiếp! thế thì có phí rượu không? Thế thì có khổ
hắn không? Không biết đứa chết mẹ nào lại đẻ ra thân hắn
cho hắn khổ đến nông-nỗi này? A hà! phải đấy, hắn cứ thế
mà chửi, hắn cứ chửi đứa chết mẹ nào đẻ ra thân hắn, đẻ ra
thằng Chí-Phèo! Hắn nghiến răng vào mà chửi cái đứa đã đẻ
ra Chí-Phèo. Nhưng mà biết đứa nào đã đẻ ra Chí-Phèo? Có
mà giời biết! Hắn không biết, cả làng V

In [44]:
import gradio as gr

c:\RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [96]:
# Launch first version
print("Launching keyword-based RAG chatbot...")
view = gr.ChatInterface(fn=chat).launch()

Launching keyword-based RAG chatbot...
* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


# 2. RAG bigger idea with vector search - optimized version

In [3]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "data/Chí_Phèo.pdf"

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print("Tổng số trang đã tải:", len(documents))
print(documents[0].page_content[:500])

Tổng số trang đã tải: 45
1
Chí Phèo
Nam Cao
1941
Được lấy từ Wikisource vào ngày 26 tháng 3, 2026


In [4]:
#Nhiều file data 
import os
import glob
from langchain_community.document_loaders import PyPDFLoader

pdf_files = glob.glob("data/*.pdf")

documents = []

for pdf_file in pdf_files:
    loader = PyPDFLoader(pdf_file)
    file_docs = loader.load()

    for doc in file_docs:
        doc.metadata["source_file"] = os.path.basename(pdf_file)

    documents.extend(file_docs)

print("Tổng số file PDF:", len(pdf_files))
print("Tổng số trang đã tải:", len(documents))

Tổng số file PDF: 2
Tổng số trang đã tải: 62


In [5]:
documents[0]

Document(metadata={'producer': 'Wikisource', 'creator': 'Wikisource', 'creationdate': '2026-03-26T08:10:09+00:00', 'author': 'Nam Cao', 'moddate': '2026-03-26T08:10:11+00:00', 'title': 'Chí Phèo', 'source': 'data\\Chí_Phèo.pdf', 'total_pages': 45, 'page': 0, 'page_label': '1', 'source_file': 'Chí_Phèo.pdf'}, page_content='1\nChí Phèo\nNam Cao\n1941\nĐược lấy từ Wikisource vào ngày 26 tháng 3, 2026')

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # Chia nhỏ hơn một chút để dễ tìm kiếm hơn.
    chunk_overlap=200,  # Giảm sự chồng chéo để cải thiện hiệu suất.
    # separators=["\n\n", "\n", ". ", " ", ""]  # Better separation
)

chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")

Created 110 chunks


In [7]:
chunks[9]

Document(metadata={'producer': 'Wikisource', 'creator': 'Wikisource', 'creationdate': '2026-03-26T08:10:09+00:00', 'author': 'Nam Cao', 'moddate': '2026-03-26T08:10:11+00:00', 'title': 'Chí Phèo', 'source': 'data\\Chí_Phèo.pdf', 'total_pages': 45, 'page': 5, 'page_label': '6', 'source_file': 'Chí_Phèo.pdf'}, page_content='6\n— Cả các ông, các bà nữa; về đi thôi chứ! Có gì mà xúm lại\nđông thế này?\nKhông ai nói gì, người ta lảng dần đi. Vì nể cụ Bá cũng có,\nnhưng vì nghĩ đến sự yên ổn của mình cũng có: người nhà\nquê vốn ghét lôi thôi; ai dại gì đứng ỳ ra đấy, có làm sao họ\nlại triệu mình đi làm chứng! Sau còn trơ lại Chí-Phèo và cha\ncon cụ Bá. Bấy giờ cụ mới lại gần hắn, khẽ lay và gọi:\n— Anh Chí ơi! sao anh lại làm ra thế?\nChí-Phèo lim rim mắt, rên lên:\n— Tao chỉ liều chết với bố con nhà mày đấy thôi. Nhưng tao\nmà chết thì có thằng sạt nghiệp, mà còn rũ tù nữa chưa biết\nchừng.\nCụ Bá cười nhạt, nhưng tiếng cười ròn-rã lắm — người ta\nbảo cụ hơn người cũng chỉ bởi cái cười:\n—

In [8]:
doc_types = set(chunk.metadata['source_file'] for chunk in chunks)
print(f"Các loại tài liệu đã tìm thấy: {', '.join(doc_types)}")

Các loại tài liệu đã tìm thấy: Chí_Phèo.pdf, Lão_Hạc.pdf


In [10]:
for chunk in chunks:
    if "Nam Cao" in chunk.page_content:
        print(chunk)
        print("-------------------")

page_content='1
Chí Phèo
Nam Cao
1941
Được lấy từ Wikisource vào ngày 26 tháng 3, 2026' metadata={'producer': 'Wikisource', 'creator': 'Wikisource', 'creationdate': '2026-03-26T08:10:09+00:00', 'author': 'Nam Cao', 'moddate': '2026-03-26T08:10:11+00:00', 'title': 'Chí Phèo', 'source': 'data\\Chí_Phèo.pdf', 'total_pages': 45, 'page': 0, 'page_label': '1', 'source_file': 'Chí_Phèo.pdf'}
-------------------
page_content='1
Lão Hạc
Nam Cao
1943
Được lấy từ Wikisource vào ngày 26 tháng 3, 2026' metadata={'producer': 'Wikisource', 'creator': 'Wikisource', 'creationdate': '2026-03-26T11:49:02+00:00', 'author': 'Nam Cao', 'moddate': '2026-03-26T11:49:03+00:00', 'title': 'Lão Hạc', 'source': 'data\\Lão_Hạc.pdf', 'total_pages': 17, 'page': 0, 'page_label': '1', 'source_file': 'Lão_Hạc.pdf'}
-------------------


In [11]:
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [10]:
print(type(chunks))
print(len(chunks))
print(chunks[0])
print(chunks[0].page_content[:300])
print(chunks[0].metadata)

<class 'list'>
110
page_content='1
Chí Phèo
Nam Cao
1941
Được lấy từ Wikisource vào ngày 26 tháng 3, 2026' metadata={'producer': 'Wikisource', 'creator': 'Wikisource', 'creationdate': '2026-03-26T08:10:09+00:00', 'author': 'Nam Cao', 'moddate': '2026-03-26T08:10:11+00:00', 'title': 'Chí Phèo', 'source': 'data\\Chí_Phèo.pdf', 'total_pages': 45, 'page': 0, 'page_label': '1', 'source_file': 'Chí_Phèo.pdf'}
1
Chí Phèo
Nam Cao
1941
Được lấy từ Wikisource vào ngày 26 tháng 3, 2026
{'producer': 'Wikisource', 'creator': 'Wikisource', 'creationdate': '2026-03-26T08:10:09+00:00', 'author': 'Nam Cao', 'moddate': '2026-03-26T08:10:11+00:00', 'title': 'Chí Phèo', 'source': 'data\\Chí_Phèo.pdf', 'total_pages': 45, 'page': 0, 'page_label': '1', 'source_file': 'Chí_Phèo.pdf'}


In [21]:
try:
    del vectorstore
except:
    pass

try:
    del embeddings
except:
    pass

import gc
gc.collect()

print("Đã giải phóng biến")

Đã giải phóng biến


In [1]:
import shutil
import os

if os.path.exists("chroma_db"):
    shutil.rmtree("chroma_db")
    print("Đã xóa toàn bộ chroma_db")
else:
    print("Không thấy chroma_db")

Đã xóa toàn bộ chroma_db


embeddings

In [12]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

DB_NAME = "chroma_db"

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

chroma_db = f"./{DB_NAME}"

if os.path.exists(chroma_db):
    Chroma(
        persist_directory=chroma_db,
        embedding_function=embeddings
    ).delete_collection()
    print("Đã xóa collection cũ")
else:
    print("Chưa có DB cũ")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=chroma_db
)

print(f"Vectorstore được tạo bằng {vectorstore._collection.count()} documents")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4919.25it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Đã xóa collection cũ
Vectorstore được tạo bằng 110 documents


Kiểm tra mỗi vecto có bao nhiêu chiều

In [13]:
# Lấy ra bộ sưu tập vector từ vectorstore
collection = vectorstore._collection

# Lấy 1 embedding từ database
sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]

# Kiểm tra số chiều (số phần tử trong vector)
dimensions = len(sample_embedding)
print(f"The vectors have {dimensions:,} dimensions")

The vectors have 384 dimensions


In [13]:
sample_embedding

array([-1.27632901e-01,  5.38610853e-02, -3.85791957e-02,  2.02306136e-02,
       -8.55069160e-02,  5.63296489e-02, -8.74113699e-04,  5.72156757e-02,
        2.99358014e-02,  7.03179613e-02,  1.22017048e-01, -1.09198041e-01,
       -3.95029150e-02, -2.11515315e-02, -2.12058388e-02, -3.88545096e-02,
       -1.15660377e-01,  7.56849954e-03, -6.21432215e-02, -5.86396083e-02,
        1.24842133e-02,  2.01356225e-02, -4.02184762e-03,  2.52442397e-02,
       -1.79682039e-02, -3.34636755e-02,  4.79451322e-04,  2.18425523e-02,
        9.89402682e-02, -1.17005510e-02,  2.44733300e-02,  1.19125038e-01,
        2.91408729e-02,  2.41378695e-02,  1.70402303e-02,  1.48783559e-02,
       -3.84317711e-02, -8.54355395e-02,  5.48167266e-02,  2.33133528e-02,
       -2.61887629e-02, -4.89021912e-02,  4.04184461e-02, -4.50862572e-02,
        5.85241914e-02,  4.17449810e-02, -7.82811046e-02,  3.20864357e-02,
       -3.65126990e-02,  1.70646273e-02, -9.96712670e-02, -6.72210660e-03,
        3.69215821e-04,  

In [14]:
# Lấy toàn bộ vector, tài liệu và metadata từ collection
result = collection.get(include=["embeddings", "documents", "metadatas"])

# Đưa embedding vào mảng numpy
vectors = np.array(result["embeddings"])

# Lưu lại văn bản
documents = result["documents"]

# Lấy loại tài liệu từ source_file
doc_types = [metadata["source_file"] for metadata in result["metadatas"]]

color_map = {
    "Chí_Phèo.pdf": "blue",
    "Lão_Hạc.pdf": "green"
}
# Nếu có file lạ thì nó không crash, mà tự cho màu gray
colors = [color_map.get(t, "gray") for t in doc_types]

In [15]:
from sklearn.manifold import TSNE
import plotly.graph_objects as go

# Giảm chiều vector xuống 2D
tsne = TSNE(
    n_components=2,
    random_state=42,
    perplexity=min(30, len(vectors) - 1)
)

reduced_vectors = tsne.fit_transform(vectors)

# Tạo biểu đồ scatter 2D
fig = go.Figure(
    data=[
        go.Scatter(
            x=reduced_vectors[:, 0],
            y=reduced_vectors[:, 1],
            mode="markers",
            marker=dict(
                size=6,
                color=colors,
                opacity=0.8
            ),
            text=[
                f"File: {t}<br>Văn bản: {d[:100]}..."
                for t, d in zip(doc_types, documents)
            ],
            hoverinfo="text"
        )
    ]
)

fig.update_layout(
    title="Biểu đồ 2D Chroma Vector Store",
    xaxis_title="TSNE Dimension 1",
    yaxis_title="TSNE Dimension 2",
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show(renderer="browser")

In [16]:
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_classic.chains import ConversationalRetrievalChain

In [17]:
# Tạo mô hình ngôn ngữ và chuỗi hội thoại truy xuất
llm = ChatGroq(
    temperature=0.7,
    model_name=MODEL
)
# Thiết lập bộ nhớ hội thoại 
memory = ConversationBufferWindowMemory(
    memory_key="chat_history",
    return_messages=True
)
# Tạo retriever từ vectorstore(Chroma)
retriever = vectorstore.as_retriever()
# Kết nối tất cả lại với nhau trong một chuỗi hội thoại truy xuất (RAG pipeline)
conversation_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory
)

C:\Users\ACER\AppData\Local\Temp\ipykernel_20344\1804977248.py:7: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(


In [18]:
# Test query with performance monitoring
import time


def test_query_performance():
    """Test query with timing"""
    query = "Bạn có thể mô tả ngắn gọn về Lão Hạc không?"
    start_time = time.time()
    result = conversation_chain.invoke({"question": query})
    end_time = time.time()
    
    print(f"Query processed in {end_time - start_time:.2f} seconds")
    print("Answer:", result["answer"])
    if "source_documents" in result:
        print(f"Used {len(result['source_documents'])} source documents")

In [19]:
test_query_performance()

Query processed in 1.15 seconds
Answer: Lão Hạc là một nhân vật trong văn học, cụ thể là trong tác phẩm "Lão Hạc" của nhà văn Nam Cao. Ông là một người nông dân già, nghèo khó, nhưng có tinh thần tự trọng và ý chí kiên cường. Lão Hạc sống một cuộc sống đơn giản, gắn bó với mảnh vườn nhỏ của mình và luôn cố gắng bảo vệ nó. Ông cũng là một người cha yêu thương con trai và muốn bảo vệ tương lai cho con mình.


In [20]:
# set up a new conversation memory for the chat
memory = ConversationBufferWindowMemory(memory_key='chat_history', return_messages=True)

# putting it together: set up the conversation chain with the GPT 4o-mini LLM, the vector store and memory
conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory)

In [21]:
# Wrapping in a function - note that history isn't used, as the memory is in the conversation_chain

def chat(message, history):
    result = conversation_chain.invoke({"question": message})
    return result["answer"]

In [22]:
import gradio as gr

view = gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [23]:
from langchain_core.callbacks import StdOutCallbackHandler
from langchain_groq import ChatGroq
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_classic.chains import ConversationalRetrievalChain

llm = ChatGroq(
    model=MODEL,
    temperature=0
)

memory = ConversationBufferWindowMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

conversation_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    callbacks=[StdOutCallbackHandler()],
    return_source_documents=True
)

query = "Người bạn thân của Lão Hạc là ai?"
result = conversation_chain.invoke({"question": query})
output_key="answer"
print("\nAnswer:", result["answer"])

print("\nSOURCE DOCS:")
for i, doc in enumerate(result["source_documents"], 1):
    print(f"\n--- Doc {i} ---")
    print(doc.metadata)
    print(doc.page_content[:400])



> Entering new ConversationalRetrievalChain chain...


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: Use the following pieces of context to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
----------------
khoăn quá thế...
Lão hút xong, đặt xe điếu xuống, quay ra ngoài, thở khói.
Sau một điếu thuốc lào, óc người ta tê dại đi trong một nỗi
đê mê nhẹ nhõm. Lão Hạc ngồi lặng lẽ, hưởng chút khoái lạc
con con ấy. Tôi cũng ngồi lặng lẽ. Tôi nghĩ đến mấy quyển
sách quý của tôi. Hồi bị ốm nặng ở Sài Gòn tôi bán gần hết
cả áo quần, nhưng vẫn không chịu bán cho ai một quyển. Ốm

Thị cười, và nói lảng:
— Hôm qua làm biên-bản., Lý Cường nghe đâu đã tốn gần
một trăm. Thiệt người lại thiệt của.
Nhưng thị nghĩ thầm:
— Sao có lúc nó hiền như đất.
Và nhớ lại những lúc ăn nằm với hắn, thị nhìn trộm bà cô,
rồi nhìn nhanh xuống bụng:

4
- Con chó là của cháu nó mua đấy

In [24]:
# create a new Chat with OpenAI
llm = ChatGroq(temperature=0.7, model_name=MODEL)

# set up the conversation memory for the chat
memory = ConversationBufferWindowMemory(memory_key='chat_history', return_messages=True)

# the retriever is an abstraction over the VectorStore that will be used during RAG; k is how many chunks to use
retriever = vectorstore.as_retriever(search_kwargs={"k": 30})

# putting it together: set up the conversation chain with the GPT 3.5 LLM, the vector store and memory
conversation_chain = ConversationalRetrievalChain.from_llm(
    llm=llm, 
    retriever=retriever, 
    memory=memory, 
    callbacks=[StdOutCallbackHandler()]
)

query = "Nhân viên nào trong công ty đã tốt nghiệp Đại học Ngoại thương?"
result = conversation_chain.invoke({"question": query})
answer = result["answer"]
print("\nAnswer:", answer)



> Entering new ConversationalRetrievalChain chain...


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: Use the following pieces of context to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
----------------
choe choé. Thị nắm cổ hắn mà dúi xuống. Chúng tỏ tình với
nhau, không cần đến những cái hôn. Ai lại hôn, khi có những
cái môi nứt nẻ như bờ ruộng vào kỳ đại hạn và cái mặt rạch
ngang rạch dọc như mặt thớt. Vả lại có những cách âu-yếm
bình dân hơn, chúng cấu véo hoặc phát nhau... thiết thực biết
mấy...
Chúng sẽ làm thành một cặp rất xứng đôi. Chúng cũng nhận
thấy thế, và nhất định là lấy nhau. Như thế năm ngày chẵn
Thị ở nhà hắn cả ngày lẫn đêm, trừ những lúc đi kiếm tiền.

khoăn quá thế...
Lão hút xong, đặt xe điếu xuống, quay ra ngoài, thở khói.
Sau một điếu thuốc lào, óc người ta tê dại đi trong một nỗi
đê mê nhẹ nhõm. Lão Hạc ngồi lặng lẽ, hưởn

In [25]:
def chat(question, history):
    result = conversation_chain.invoke({"question": question})
    return result["answer"]

In [26]:
view = gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.




> Entering new ConversationalRetrievalChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, in its original language.

Chat History:

Human: Nhân viên nào trong công ty đã tốt nghiệp Đại học Ngoại thương?
Assistant: Tôi không biết. Tôi không có thông tin về công ty hoặc nhân viên của công ty. Văn bản trên dường như là một đoạn trích từ một tác phẩm văn học và không cung cấp thông tin về công ty hoặc nhân viên. Nếu bạn cần tìm kiếm thông tin khác, vui lòng cho tôi biết!
Follow Up Input: lão hạt là ai
Standalone question:

> Finished chain.


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: Use the following pieces of context to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
----------------
1
Lão Hạc
Nam Cao
1943